<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/Lunar_Tithi_Calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](http://1.bp.blogspot.com/_5QlanosqxiQ/SXLOLTu7S7I/AAAAAAAAAm8/0r6C7lCxsic/S1600-R/p21-header-003.png) <br>


---

Website - [Parashar21](https://parashar21.blogspot.com)

## Lunar Tithi on a given Date

###Install Swiss Ephemeris <br>

https://pypi.org/project/pyswisseph/  - Python Package Documentation <br>
https://www.astro.com/swisseph/swephinfo_e.htm <br>
https://www.astro.com/swisseph/swephprg.htm - Programmer Documentation <br>
https://pythonhosted.org/pyswisseph/swisseph-module.html

In [1]:
!pip install pyswisseph > /dev/null

In [2]:
#!wget -q http://www.astro.com/ftp/swisseph/ephe/seas_18.se1
#!wget -q http://www.astro.com/ftp/swisseph/ephe/semo_18.se1
#!wget -q http://www.astro.com/ftp/swisseph/ephe/sepl_18.se1

#Download Swiss Ephemeris Files
#!wget  http://www.astro.com/ftp/swisseph/ephe/seas_18.se1
!wget -q https://github.com/aloistr/swisseph/raw/master/ephe/seas_18.se1
#!wget  http://www.astro.com/ftp/swisseph/ephe/semo_18.se1
!wget -q https://github.com/aloistr/swisseph/raw/master/ephe/semo_18.se1
#!wget  http://www.astro.com/ftp/swisseph/ephe/sepl_18.se1
!wget  -q https://github.com/aloistr/swisseph/raw/master/ephe/sepl_18.se1

!mkdir ephe
!mv *.se1 ephe

mkdir: cannot create directory ‘ephe’: File exists


In [3]:
import swisseph as swe
swe.version

'2.10.03'

In [4]:
from datetime import datetime
import pytz



In [14]:
# --- 1. System & Ephemeris Configuration ---
swe.set_ephe_path('/content/ephe')

# Calendar and Ayanamsha setup (1 = Lahiri)
SE_GREG_CAL = 1
gregflag = SE_GREG_CAL
swe.set_sid_mode(1)

# Flag combination
SEFLG_SPEED = int(256)
SEFLG_SIDEREAL = int(64*1024)
iflag = SEFLG_SPEED | SEFLG_SIDEREAL

In [15]:
# --- 2. Calculate Julian Day for Current Time in Kolkata ---
#now = datetime.now(pytz.timezone("Asia/Kolkata"))
#now_utc = now.astimezone(pytz.utc)

#year   = now_utc.year
#month  = now_utc.month
#day    = now_utc.day
#hour   = now_utc.hour
#minute = now_utc.minute
#second = now_utc.second + now_utc.microsecond / 1e6

# Get Julian Day (Using jd_ut for planetary calculations)
#jd_ut, jd_et = swe.utc_to_jd(year, month, day, hour, minute, second, swe.GREG_CAL)
#print(jd_ut, jd_et)


### Tithi is determined for a point of time
To get the tithi for "a day" we usually determine the tithi at sunrise. So the first task is to get the time of sunrise for a particular day

In [16]:
from datetime import datetime
import pytz
import requests

# --- 1. Give your actual date string here ---
# Example: "2026-10-23"
targetDate = "2026-10-23"

# Set target timezone to Asia/Kolkata
tz_kolkata = pytz.timezone("Asia/Kolkata")

# Parse the YYYY-MM-DD string into a naive date object, then localize it to 00:00 AM midnight
parsed_targetDate = datetime.strptime(targetDate, "%Y-%m-%d")
localtargetDate = tz_kolkata.localize(parsed_targetDate)

# Extract the date string for the API parameter
targetDatestr = localtargetDate.strftime("%Y-%m-%d")

# --- 2. REST API Request for Calcutta Lat/Long ---
target_lat = 22.5726
target_lng = 88.3639

#url = f"https://api.sunrisesunset.io{calcutta_lat}&lng={calcutta_lng}&date={date_str}"
url = f"https://api.sunrisesunset.io/json?lat={target_lat}&lng={target_lng}&date={targetDatestr}"


try:
    # Fetch data directly from the endpoint
    response = requests.get(url).json()
    api_sunrise_time = response["results"]["sunrise"]  # e.g., "5:38:12 AM"

    # --- 3. Parse Response into your custom Target Date Tuple ---
    combined_str = f"{targetDatestr} {api_sunrise_time}"
    parsed_dt = datetime.strptime(combined_str, "%Y-%m-%d %I:%M:%S %p")

    # Formats to your exact tuple structure: (Year, Month, Day, Hour, Minute)
    sunrise_target_format = (
        parsed_dt.year,
        parsed_dt.month,
        parsed_dt.day,
        parsed_dt.hour,
        parsed_dt.minute
    )

    print(f"Input Date          : {targetDatestr}")
    print(f"API String Response : {api_sunrise_time}")
    print(f"Sunrise Tuple Output: {sunrise_target_format}")

except Exception as e:
    print(f"Execution Error: {e}")


Input Date          : 2026-10-23
API String Response : 5:35:53 AM
Sunrise Tuple Output: (2026, 10, 23, 5, 35)


The sunrise time is converted to julian date time format for Swiss Ephemeris

In [17]:
# --- 2. Configuration: Set to None for 'now', or provide a specific (Year, Month, Day, Hour, Minute) tuple
# Examples:
# target_date_input = None                   # For current time
#target_date_input = (2026, 10, 25, 15, 19)  # For YYYY,MM,DD,HH,MM
target_date_input = sunrise_target_format    # at sunrise from last cell

# Define the local timezone
tz_kolkata = pytz.timezone("Asia/Kolkata")

if target_date_input is None:
    # Option A: Set to current time right now
    localDateTimeStr = datetime.now(tz_kolkata)
    print("None")
else:
    # Option B: Set to a specific target date/time
    year_in, month_in, day_in, hour_in, min_in = target_date_input
    local_date = datetime(year_in, month_in, day_in, hour_in, min_in, 0)
    localDateTimeStr = tz_kolkata.localize(local_date)
    print(localDateTimeStr)



print( sunrise_target_format, localDateTimeStr)

# Convert to UTC exactly as your script expects
localDateTimeStr_utc = localDateTimeStr.astimezone(pytz.utc)

# Extract date-time components for Swiss Ephemeris
year   = localDateTimeStr_utc.year
month  = localDateTimeStr_utc.month
day    = localDateTimeStr_utc.day
hour   = localDateTimeStr_utc.hour
minute = localDateTimeStr_utc.minute
second = localDateTimeStr_utc.second + localDateTimeStr_utc.microsecond / 1e6

# Get Julian Day (Using jd_ut for planetary calculations)
jd_ut, jd_et = swe.utc_to_jd(year, month, day, hour, minute, second, swe.GREG_CAL)
print(jd_ut, jd_et)


2026-10-23 05:35:00+05:30
(2026, 10, 23, 5, 35) 2026-10-23 05:35:00+05:30
2461336.504272963 2461336.5034765075


Using Julian Dates, we get Sun, Moon Longitudes

In [18]:
# --- 3. Planetary Calculation Loop (Using your exact indexing layout) ---
bLonS = []
bSpeedS = []
bRetS = []

# Explicit target body list matching your snippet layout
target_bodies = [0, 1, 2, 3, 4, 5, 6, 11]

for body in target_bodies:
    pData = swe.calc_ut(jd_ut, body, iflag)

    # Matching your exact nested tuple corrections: pData[0][index]
    bLonS.append(pData[0][0])    # Longitude position
    bSpeedS.append(pData[0][3])  # Speed in longitude (deg/day)

    if pData[0][3] >= 0:
        bRetS.append('False')
    else:
        bRetS.append('True')

In [19]:
# --- 4. Tithi Engine ---


# A Tithi represents a 12° increase in the Moon's angular separation
# from the Sun, measured along the ecliptic. The 360° lunar cycle is
# divided into 30 Tithis: 15 in Shukla Paksha (waxing Moon) and 15 in
# Krishna Paksha (waning Moon). Elongation is calculated modulo 360°
# to handle the transition across 0°. Shukla Pratipada begins at 0°,
# Purnima occupies 168°–180°, and Amavasya occupies 348°–360°.
# The Tithi number and traditional name are derived from elongation.
# The remaining duration is estimated from the angular distance to
# the next 12° boundary and the Moon's velocity relative to the Sun,
# assuming constant angular velocities (degrees/day). The resulting
# time in hours is approximate, not an exact Tithi ending time.


# Because body 0 (Sun) is processed first, it is at index 0
# Because body 1 (Moon) is processed second, it is at index 1

sun_lon = bLonS[0]
moon_lon = bLonS[1]

sun_speed = bSpeedS[0]
moon_speed = bSpeedS[1]

# Step A: Compute Elongation (Angular separation between Moon and Sun)
elongation = (moon_lon - sun_lon + 360.0) % 360.0

# Step B: Compute Absolute Tithi Index (1 to 30)
tithi_index = int(elongation // 12.0) + 1

# Step C: Determine Paksha (Waxing / Waning Phase)
paksha = "Shukla" if tithi_index <= 15 else "Krishna"

# Step D: Determine relative Tithi number inside its Paksha (1 to 15)
tithi_in_paksha = tithi_index if tithi_index <= 15 else tithi_index - 15

# Step E: Traditional Tithi names mapping
tithi_names = [
    "Pratipada", "Dwitiya", "Tritiya", "Chaturthi", "Panchami",
    "Shashthi", "Saptami", "Ashtami", "Navami", "Dashami",
    "Ekadashi", "Dwadashi", "Trayodashi", "Chaturdashi",
    "Purnima" if paksha == "Shukla" else "Amavasya"
]
current_tithi_name = tithi_names[tithi_in_paksha - 1]

# Step F: Calculate Hours Remaining in this Tithi
degrees_passed_in_current_tithi = elongation % 12.0
degrees_left_in_tithi = 12.0 - degrees_passed_in_current_tithi
net_speed = moon_speed - sun_speed
hours_remaining = (degrees_left_in_tithi / net_speed) * 24.0




In [20]:
# --- 5. Output Results ---
print("=== TITHI CALCULATION SUMMARY ===")
#print(f"Current Local Time (Kolkata): {now.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Target Date: {targetDatestr}")
print(f"Sun Longitude  : {sun_lon:.4f}°")
print(f"Moon Longitude : {moon_lon:.4f}°")
print(f"Net Elongation : {elongation:.4f}°")
print("-" * 33)
print(f"Paksha         : {paksha}")
print(f"Absolute Index : Tithi #{tithi_index} out of 30")
print(f"Current Tithi  : {paksha} Paksha - {current_tithi_name} ({tithi_in_paksha})")
print(f"Time Remaining : ~{hours_remaining:.2f} hours left in this tithi")

=== TITHI CALCULATION SUMMARY ===
Target Date: 2026-10-23
Sun Longitude  : 185.3711°
Moon Longitude : 324.7824°
Net Elongation : 139.4113°
---------------------------------
Paksha         : Shukla
Absolute Index : Tithi #12 out of 30
Current Tithi  : Shukla Paksha - Dwadashi (12)
Time Remaining : ~9.06 hours left in this tithi


In [21]:
!date

Sun Oct 25 07:59:09 AM UTC 2026


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [Finished scrolling? Now try reading a book. ](http://chronos.yantrajaal.com) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)